# Model training for splice-site-predictor using prepared data

In [5]:
# install pytorch
# !pip install torch
!pip install scikit-learn

  Obtaining dependency information for scikit-learn from https://files.pythonhosted.org/packages/8d/da/4810a28e473185429e45a57eebcc91fc991b33d889cc0676063e671db03d/scikit_learn-1.9.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for scipy>=1.10.0 from https://files.pythonhosted.org/packages/09/7d/af933f0f6e0767995b4e2d705a0665e454d1c19402aa7e895de3951ebb04/scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 499.2 kB/s eta 0:00:00m eta 0:00:01:--
  Obtaining dependency information for joblib>=1.4.0 from https://files.pythonhosted.org/packages/7b/91/984aca2ec129e2757d1e4e3c81c3fcda9d0f85b74670a094cc443d9ee949/joblib-1.5.3-py3-none-any.whl.metadata
  Obtaining dependency information for narwhals>=2.0.1 from https://files.pythonhosted.org/packages/48/ca/36339329c4604adbcc99c899b7eb1ce1a555c499b6a6860757dc9bfed36d/narwhals-2.22.1-py3-none-any.whl.

In [7]:
# load datset (combine positives and negatives)

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Load one-hot encoded files
positives = np.load('data/positives.npy')   # shape: (150000, 400)
negatives = np.load('data/negatives.npy')    # shape: (150000, 400)

# Create labels
# 1 = real splice donor site, 0 = non-donor GT dinucleotide
pos_labels = np.ones(len(positives),  dtype=np.float32)  # 150000 ones
neg_labels = np.zeros(len(negatives), dtype=np.float32)  # 150000 zeros

# Combine into one dataset
X = np.concatenate([positives, negatives], axis=0)  # shape: (300000, 400)
y = np.concatenate([pos_labels, neg_labels], axis=0) # shape: (300000,)

print(f"X shape: {X.shape}")  # (300000, 400)
print(f"y shape: {y.shape}")  # (300000,)
print(f"Class balance: {y.mean():.2f}")  # should be 0.50 → balanced

X shape: (310028, 400)
y shape: (310028,)
Class balance: 0.50


In [8]:
# Split into train / validation / test set

# Seperate the test set
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.10,       # 10% → 30,000 samples for testing
    random_state=42,      # reproducibility
    stratify=y            # keeps 50/50 class balance in each split
)

# Split remaining data into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.111,      # ~10% of total --> 30,000 validation samples
    random_state=42,
    stratify=y_trainval
)

# Check sizes
print(f"Train:      {X_train.shape[0]:>7,} samples")  # ~240,000
print(f"Validation: {X_val.shape[0]:>7,} samples")    # ~30,000
print(f"Test:       {X_test.shape[0]:>7,} samples")   # ~30,000

Train:      248,053 samples
Validation:  30,972 samples
Test:        31,003 samples


In [9]:
# create Pytorch data set

class SpliceSiteDataset(Dataset):
    """
    A PyTorch Dataset for splice donor site classification.
    Stores one-hot encoded sequences and their binary labels.
    """
    def __init__(self, X, y):
        # Convert numpy arrays to PyTorch tensors once (efficient)
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        """Returns total number of samples — used by DataLoader."""
        return len(self.X)

    def __getitem__(self, idx):
        """Returns one sample by index — called by DataLoader."""
        return self.X[idx], self.y[idx]


# Set up the 3 datasets
train_dataset = SpliceSiteDataset(X_train, y_train)
val_dataset   = SpliceSiteDataset(X_val,   y_val)
test_dataset  = SpliceSiteDataset(X_test,  y_test)

# Sanity check
sample_X, sample_y = train_dataset[0]
print(f"One sample — X shape: {sample_X.shape}, label: {sample_y}")
# X shape: torch.Size([400]), label: tensor(1.)

One sample — X shape: torch.Size([400]), label: 1.0


In [10]:
# Create data loaders

BATCH_SIZE = 256  # number of samples processed together in one forward pass

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # ← shuffle ONLY training data
    num_workers=4,      # parallel data loading (use 0 on Windows if issues)
    pin_memory=True     # faster GPU transfer
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,      # ← no shuffling for val/test (order doesn't matter)
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# --- Inspect one batch ---
X_batch, y_batch = next(iter(train_loader))
print(f"Batch X shape: {X_batch.shape}")  # torch.Size([256, 400])
print(f"Batch y shape: {y_batch.shape}")  # torch.Size([256])

/home/sarah_rinke/miniconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Batch X shape: torch.Size([256, 400])
Batch y shape: torch.Size([256])


In [11]:
# Reshape for CNN

class SpliceSiteDataset(Dataset):
    def __init__(self, X, y, reshape_for_cnn=True):
        X_tensor = torch.tensor(X, dtype=torch.float32)
        
        if reshape_for_cnn:
            # (N, 400) --> (N, 100, 4) --> (N, 4, 100)
            # Encoding order: [A,C,G,T] repeated per position
            X_tensor = X_tensor.view(-1, 100, 4).permute(0, 2, 1)
        
        self.X = X_tensor
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Shape check:
# Flat MLP input:  torch.Size([256, 400])
# CNN input:       torch.Size([256, 4, 100]) # batch, channels, sequence_length

In [13]:
# install matplotlib if necessary
!pip install matplotlib

  Obtaining dependency information for matplotlib from https://files.pythonhosted.org/packages/53/f4/f0b4f9ba7ec14a7af8151f3ad71ecfe3561e6ba38cfab1db3681ba4ca112/matplotlib-3.11.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 1.3 MB/s eta 0:00:001.9 MB/s eta 0:00:01
  Obtaining dependency information for contourpy>=1.0.1 from https://files.pythonhosted.org/packages/5f/4b/6157f24ca425b89fe2eb7e7be642375711ab671135be21e6faa100f7448c/contourpy-1.3.3-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for cycler>=0.10 from https://files.pythonhosted.org/packages/e7/05/c19819d5e3d95294a6f5947fb9b9629efb316b96de511b418c53d245aae6/cycler-0.12.1-py3-none-any.whl.metadata
  Obtaining dependency information for fonttools>=4.22.0 from https://files.pythonhosted.org/packages/0b/43/a81f20050a3115b57d62c8e781446949512eac36690dc384ccea65ff4cc1/fonttools-4.63.0-cp311-cp3

In [14]:
# Setup
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Use GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu
